# import functinos

In [ ]:
import math
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error
import tensorflow as tf
import os
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Import data

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Austria.xlsx to Austria.xlsx
Saving Belgium.xlsx to Belgium.xlsx
Saving Denmark.xlsx to Denmark.xlsx
Saving Finland.xlsx to Finland.xlsx
Saving France.xlsx to France.xlsx
Saving Germany.xlsx to Germany.xlsx
Saving Greece.xlsx to Greece.xlsx
Saving Ireland.xlsx to Ireland.xlsx
Saving Italy.xlsx to Italy.xlsx
Saving Luxembourg.xlsx to Luxembourg.xlsx
Saving Netherlands.xlsx to Netherlands.xlsx
Saving Portugal.xlsx to Portugal.xlsx
Saving Spain.xlsx to Spain.xlsx
Saving Sweden.xlsx to Sweden.xlsx


In [ ]:
austria = pd.read_excel("Austria.xlsx")
belgium = pd.read_excel('Belgium.xlsx')
denmark = pd.read_excel('Denmark.xlsx')
finland = pd.read_excel('Finland.xlsx')
france = pd.read_excel('France.xlsx')
germany = pd.read_excel('Germany.xlsx')
ireland = pd.read_excel('Ireland.xlsx')
italy = pd.read_excel('Italy.xlsx')
luxembourg = pd.read_excel('Luxembourg.xlsx')
netherlands = pd.read_excel('Netherlands.xlsx')
portugal = pd.read_excel('Portugal.xlsx')
spain = pd.read_excel('Spain.xlsx')
sweden = pd.read_excel('Sweden.xlsx')

# Functions

In [ ]:
def fit_lstm_on_series(
    train_series_1d: np.ndarray,
    lookback: int,
    units: int,
    dropout: float,
    epochs: int = 200,
    batch_size: int = 16,
    patience: int = 20,
    verbose: int = 0,
    seed: int = 42
):
    """
    train_series_1d: 1D numpy array of length n
    Returns: (scaler, model)
    """
    np.random.seed(seed)
    tf.random.set_seed(seed)

    tf.keras.backend.clear_session() #helps clear ram

    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_series_1d.reshape(-1, 1))  # (n,1)

    # supervised windows
    X, y = [], []
    for i in range(lookback, len(train_scaled)):
        X.append(train_scaled[i - lookback:i, :])  # (lookback,1)
        y.append(train_scaled[i, :])               # (1,)
    X = np.array(X)
    y = np.array(y)

    if X.shape[0] == 0:
        raise ValueError(f"Not enough samples for lookback={lookback}")

    model = Sequential([
        Input(shape=(lookback, 1)),
        LSTM(units),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")

    es = EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)

    model.fit(
        X, y,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[es],
        verbose=verbose,
        shuffle=False
    )

    return scaler, model


def recursive_forecast(
    model,
    seed_window_scaled: np.ndarray,
    h: int
) -> np.ndarray:
    """
    recursive forecast from model
    seed_window_scaled: numpy array shape (lookback, 1) in scaled space
    Returns: preds_scaled shape (h, 1)
    """
    window = seed_window_scaled.copy()
    preds = []
    lookback = window.shape[0]

    for _ in range(h):
        x_in = window.reshape(1, lookback, 1)
        yhat = model.predict(x_in, verbose=0)[0, 0]
        preds.append(yhat)
        window = np.vstack([window[1:, :], [[yhat]]])

    return np.array(preds).reshape(-1, 1)


def make_supervised_arrays(series_1d: np.ndarray, lookback: int):
    """
    Build supervised learning arrays from a 1D series.
    Returns X, y in the original scale.
    """
    series_1d = np.asarray(series_1d, dtype=float).reshape(-1)
    X, y = [], []
    for i in range(lookback, len(series_1d)):
        X.append(series_1d[i - lookback:i].reshape(-1, 1))
        y.append(series_1d[i])
    if len(X) == 0:
        return np.empty((0, lookback, 1)), np.empty((0,))
    return np.array(X), np.array(y)


def fit_lstm_and_get_validation_residuals(
    train_series_1d: np.ndarray,
    lookback: int,
    units: int,
    dropout: float,
    epochs: int = 200,
    batch_size: int = 16,
    patience: int = 20,
    verbose: int = 0,
    seed: int = 42,
    val_fraction: float = 0.2
):
    """
    Fit the final chosen hyperparameters on an explicit train/validation split
    and return validation residuals for PI construction.

    Returns:
      val_residuals, val_actual, val_pred, n_train_inner, n_val
    """
    series = np.asarray(train_series_1d, dtype=float).reshape(-1)
    n_train = len(series)
    n_val = max(1, math.ceil(val_fraction * n_train))
    n_train_inner = n_train - n_val

    if n_train_inner <= lookback:
        raise ValueError(
            f"Training too small for explicit validation with lookback={lookback}. "
            f"n_train={n_train}, n_train_inner={n_train_inner}, n_val={n_val}"
        )

    train_inner = series[:n_train_inner]
    val_actual = series[n_train_inner:]

    scaler, model = fit_lstm_on_series(
        train_series_1d=train_inner,
        lookback=lookback,
        units=units,
        dropout=dropout,
        epochs=epochs,
        batch_size=batch_size,
        patience=patience,
        verbose=verbose,
        seed=seed
    )

    train_inner_scaled = scaler.transform(train_inner.reshape(-1, 1))
    seed_window = train_inner_scaled[-lookback:, :]
    val_preds_scaled = recursive_forecast(model, seed_window, h=n_val)
    val_pred = scaler.inverse_transform(val_preds_scaled).ravel()
    val_residuals = val_actual - val_pred

    return val_residuals, val_actual, val_pred, n_train_inner, n_val


def calculate_training_mape(
    model,
    scaler,
    train_series_1d: np.ndarray,
    lookback: int
):
    """
    Calculate in-sample training MAPE for the final model only.
    """
    X_train, y_train_true = make_supervised_arrays(train_series_1d, lookback)
    if len(y_train_true) == 0:
        return np.nan

    n_samples = X_train.shape[0]
    X_scaled = np.empty_like(X_train, dtype=float)
    for i in range(n_samples):
        X_scaled[i] = scaler.transform(X_train[i])

    y_train_pred_scaled = model.predict(X_scaled, verbose=0)
    y_train_pred = scaler.inverse_transform(y_train_pred_scaled).ravel()

    denom = np.where(np.abs(y_train_true) < 1e-8, np.nan, y_train_true)
    train_mape = np.nanmean(np.abs((y_train_true - y_train_pred) / denom))
    return train_mape


def bootstrap_prediction_intervals(
    point_forecasts: np.ndarray,
    residuals: np.ndarray,
    n_boot: int = 500,
    alpha: float = 0.05,
    seed: int = 42
):
    """
    Empirical residual bootstrap for test-set point forecasts.
    This is only run once per country, for the final chosen model.
    """
    point_forecasts = np.asarray(point_forecasts, dtype=float).reshape(-1)
    residuals = np.asarray(residuals, dtype=float).reshape(-1)

    if len(point_forecasts) == 0:
        raise ValueError("point_forecasts is empty.")
    if len(residuals) == 0:
        raise ValueError("residuals is empty; cannot bootstrap prediction intervals.")

    rng = np.random.default_rng(seed)
    sims = np.empty((n_boot, len(point_forecasts)), dtype=float)

    for b in range(n_boot):
        sampled_residuals = rng.choice(residuals, size=len(point_forecasts), replace=True)
        sims[b, :] = point_forecasts + sampled_residuals

    lower = np.quantile(sims, alpha / 2.0, axis=0)
    upper = np.quantile(sims, 1.0 - alpha / 2.0, axis=0)
    widths = upper - lower

    return lower, upper, widths


# ------------------------------------------------------------
# Tune: last 20% of TRAIN (pre-test) used as validation
# ------------------------------------------------------------
def tune_lstm(
    df: pd.DataFrame,
    year_col: str,
    target_col: str,
    test_start_year: int,
    lookback_grid=(1, 2, 3, 4, 5),
    units_grid=(2, 4, 8, 16, 32),
    dropout_grid=(0.1, 0.2, 0.3),
    epochs: int = 100,
    batch_size: int = 16,
    patience: int = 15,
    seed: int = 42,
    verbose: int = 0
):
    """
    Returns:
      best_params (dict) OR None if everything failed,
      tuning_df (DataFrame) with val_mape for each combo.
    """
    d = df.copy()
    if year_col not in d.columns or target_col not in d.columns:
        raise ValueError(f"Missing columns: {year_col}, {target_col}")

    d = d.sort_values(year_col).reset_index(drop=True)
    d[year_col] = pd.to_numeric(d[year_col], errors="coerce")
    d[target_col] = pd.to_numeric(d[target_col], errors="coerce")
    d = d.dropna(subset=[year_col, target_col]).reset_index(drop=True)
    d[year_col] = d[year_col].astype(int)
    d = d.drop_duplicates(subset=[year_col], keep="last").reset_index(drop=True)

    train_df = d[d[year_col] < test_start_year].copy()
    if len(train_df) == 0:
        raise ValueError(f"No training data before test_start_year={test_start_year}")

    n_train = len(train_df)
    n_val = max(1, math.ceil(0.20 * n_train))
    n_train_inner = n_train - n_val
    if n_train_inner <= 0:
        raise ValueError(f"Training too small for 20% validation: n_train={n_train}, n_val={n_val}")

    train_inner = train_df.iloc[:n_train_inner]
    val_inner = train_df.iloc[n_train_inner:]
    h_val = len(val_inner)

    train_series = train_inner[target_col].to_numpy().astype(float)
    val_true = val_inner[target_col].to_numpy().astype(float)

    rows = []
    for lookback in lookback_grid:
        if len(train_series) <= lookback:
            rows.append({
                "lookback": lookback, "units": None, "dropout": None,
                "val_mape": np.nan, "error": "insufficient_train_for_lookback",
                "n_train_inner": len(train_inner), "n_val": h_val
            })
            continue

        for units in units_grid:
            for dropout in dropout_grid:
                try:
                    scaler, model = fit_lstm_on_series(
                        train_series_1d=train_series,
                        lookback=lookback,
                        units=units,
                        dropout=dropout,
                        epochs=epochs,
                        batch_size=batch_size,
                        patience=patience,
                        verbose=verbose,
                        seed=seed
                    )

                    train_inner_scaled = scaler.transform(train_series.reshape(-1, 1))
                    seed_window = train_inner_scaled[-lookback:, :]

                    preds_scaled = recursive_forecast(model, seed_window, h_val)
                    preds = scaler.inverse_transform(preds_scaled).ravel()

                    denom = np.where(np.abs(val_true) < 1e-8, np.nan, val_true)
                    val_mape = np.nanmean(np.abs((val_true - preds) / denom))

                    rows.append({
                        "lookback": lookback,
                        "units": units,
                        "dropout": dropout,
                        "val_mape": val_mape,
                        "n_train_inner": len(train_inner),
                        "n_val": h_val
                    })
                except Exception as e:
                    rows.append({
                        "lookback": lookback,
                        "units": units,
                        "dropout": dropout,
                        "val_mape": np.nan,
                        "error": str(e),
                        "n_train_inner": len(train_inner),
                        "n_val": h_val
                    })

    tuning_df = pd.DataFrame(rows)
    valid = tuning_df.dropna(subset=["val_mape"]).copy()
    if valid.empty:
        return None, tuning_df

    best = valid.sort_values("val_mape").iloc[0]
    best_params = {
        "lookback": int(best["lookback"]),
        "units": int(best["units"]),
        "dropout": float(best["dropout"])
    }
    return best_params, tuning_df


# ------------------------------------------------------------
# Run across countries: tune per-country, refit on full TRAIN, evaluate TEST
# Training MAPE and PIs are computed ONLY for the final chosen model.
# ------------------------------------------------------------
def run_all_countries_recursive_tuned(
    country_dfs: dict,
    year_col: str = "Year",
    target_col: str = "Population",
    test_start_year: int = 2012,
    lookback_grid=(1, 2, 3, 4, 5),
    units_grid=(2, 4, 8, 16, 32),
    dropout_grid=(0.1, 0.2, 0.3),
    tune_epochs: int = 100,
    final_epochs: int = 200,
    batch_size: int = 16,
    tune_patience: int = 15,
    final_patience: int = 25,
    seed: int = 42,
    verbose_tune: int = 0,
    verbose_final: int = 1,
    skip_errors: bool = True,
    n_boot: int = 500,
    alpha: float = 0.05
):
    """
    Returns:
      metrics_df: per-country TEST metrics + best hyperparams + training MAPE + PI widths
      preds_df: stacked TEST predictions + prediction intervals
      tuning_df: stacked tuning results for audit
      errors_df: countries that failed with error messages
    """
    rows = []
    preds_all = []
    tuning_all = []
    errs = []

    for country, df in country_dfs.items():
        print(f"Country: {country}")
        try:
            # --- tune on TRAIN only ---
            best_params, tuning_df = tune_lstm(
                df=df,
                year_col=year_col,
                target_col=target_col,
                test_start_year=test_start_year,
                lookback_grid=lookback_grid,
                units_grid=units_grid,
                dropout_grid=dropout_grid,
                epochs=tune_epochs,
                batch_size=batch_size,
                patience=tune_patience,
                seed=seed,
                verbose=verbose_tune
            )
            tuning_df = tuning_df.copy()
            tuning_df["Country"] = country
            tuning_df["target_col"] = target_col
            tuning_all.append(tuning_df)

            if best_params is None:
                errs.append({"Country": country, "Error": "tuning_failed_all_configs", "target_col": target_col})
                continue

            # --- clean/split for final fit + test evaluation ---
            d = df.copy()
            if year_col not in d.columns or target_col not in d.columns:
                raise ValueError(f"Missing columns: {year_col}, {target_col}")

            d = d.sort_values(year_col).reset_index(drop=True)
            d[year_col] = pd.to_numeric(d[year_col], errors="coerce")
            d[target_col] = pd.to_numeric(d[target_col], errors="coerce")
            d = d.dropna(subset=[year_col, target_col]).reset_index(drop=True)
            d[year_col] = d[year_col].astype(int)
            d = d.drop_duplicates(subset=[year_col], keep="last").reset_index(drop=True)

            train_df = d[d[year_col] < test_start_year].copy()
            test_df = d[d[year_col] >= test_start_year].copy()

            if len(test_df) == 0:
                raise ValueError("Test set is empty after cleaning.")
            if len(train_df) <= best_params["lookback"]:
                raise ValueError("Not enough training rows for chosen lookback after cleaning.")

            train_series_full = train_df[target_col].to_numpy().astype(float)
            y_true = test_df[target_col].to_numpy().astype(float)
            years_test = test_df[year_col].to_numpy()

            # --- fit final model on FULL TRAIN with best hyperparams ---
            scaler, model = fit_lstm_on_series(
                train_series_1d=train_series_full,
                lookback=best_params["lookback"],
                units=best_params["units"],
                dropout=best_params["dropout"],
                epochs=final_epochs,
                batch_size=batch_size,
                patience=final_patience,
                verbose=verbose_final,
                seed=seed
            )

            # --- training MAPE for FINAL model only ---
            training_mape = calculate_training_mape(
                model=model,
                scaler=scaler,
                train_series_1d=train_series_full,
                lookback=best_params["lookback"]
            )

            # --- validation residuals for FINAL chosen hyperparameters only ---
            val_residuals, val_actual, val_pred, n_train_inner, n_val = fit_lstm_and_get_validation_residuals(
                train_series_1d=train_series_full,
                lookback=best_params["lookback"],
                units=best_params["units"],
                dropout=best_params["dropout"],
                epochs=final_epochs,
                batch_size=batch_size,
                patience=final_patience,
                verbose=0,
                seed=seed
            )

            # --- recursive multi-step forecast across the whole test horizon ---
            train_scaled_full = scaler.transform(train_series_full.reshape(-1, 1))
            seed_window = train_scaled_full[-best_params["lookback"]:, :]
            preds_scaled = recursive_forecast(model, seed_window, h=len(test_df))
            y_pred = scaler.inverse_transform(preds_scaled).ravel()

            # --- PIs for FINAL model only ---
            pi_lower, pi_upper, pi_width = bootstrap_prediction_intervals(
                point_forecasts=y_pred,
                residuals=val_residuals,
                n_boot=n_boot,
                alpha=alpha,
                seed=seed
            )

            # --- metrics ---
            denom = np.where(np.abs(y_true) < 1e-8, np.nan, y_true)
            mape = np.nanmean(np.abs((y_true - y_pred) / denom))

            width_1st = float(pi_width[0]) if len(pi_width) >= 1 else np.nan
            width_5th = float(pi_width[4]) if len(pi_width) >= 5 else np.nan
            width_final = float(pi_width[-1]) if len(pi_width) >= 1 else np.nan

            rows.append({
                "Country": country,
                "target_col": target_col,
                "MAPE": mape,
                "Training_MAPE": training_mape,
                "n_train": len(train_df),
                "n_test": len(test_df),
                "best_lookback": best_params["lookback"],
                "best_units": best_params["units"],
                "best_dropout": best_params["dropout"],
                "n_val_residuals": len(val_residuals),
                "PI_width_1st": width_1st,
                "PI_width_5th": width_5th,
                "PI_width_final": width_final
            })

            preds_country = pd.DataFrame({
                year_col: years_test,
                f"{target_col}_true": y_true,
                f"{target_col}_pred": y_pred,
                f"{target_col}_pi_lower": pi_lower,
                f"{target_col}_pi_upper": pi_upper,
                f"{target_col}_pi_width": pi_width,
                "Country": country,
                "target_col": target_col
            })
            preds_all.append(preds_country)

        except Exception as e:
            if skip_errors:
                errs.append({"Country": country, "Error": str(e), "target_col": target_col})
            else:
                raise

    metrics_df = pd.DataFrame(rows)
    if not metrics_df.empty:
        if "Training_MAPE" in metrics_df.columns:
            metrics_df["Median_Training_MAPE"] = metrics_df["Training_MAPE"].median()
        if "MAPE" in metrics_df.columns:
            metrics_df = metrics_df.sort_values("MAPE").reset_index(drop=True)

    preds_df = pd.concat(preds_all, axis=0).reset_index(drop=True) if preds_all else pd.DataFrame()
    tuning_df = pd.concat(tuning_all, axis=0).reset_index(drop=True) if tuning_all else pd.DataFrame()
    errors_df = pd.DataFrame(errs)

    return metrics_df, preds_df, tuning_df, errors_df


# Run the code

In [ ]:
countries = {"Austria": austria, "Belgium": belgium, "Denmark": denmark,
             "Finland": finland, "France": france, "Germany": germany,
             'Ireland': ireland, 'Italy': italy, 'Luxembourg': luxembourg,
             'Netherlands': netherlands, 'Portugal': portugal, 'Spain': spain,
             'Sweden': sweden}

#sets seed and ensures reproducability
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" #uses CPU instead of GPU which is more deterministic apparently, parallasiation on GPU can vary results?


metrics_df, preds_df, tuning_df, errors_df = run_all_countries_recursive_tuned(
    countries,
    year_col="Year",
    target_col="exports",        #change this to covariate
    test_start_year=2012,    #ensure this is correct for covariate
    lookback_grid=[1,2,3,4,5],
    units_grid=[4, 8, 16],
    dropout_grid=[0.1,0.2,0.3],
    tune_epochs=100,
    final_epochs=200,
    verbose_tune=0,
    verbose_final=1
)

print(metrics_df)
print(errors_df.head())
# optional: save outputs
#metrics_df.to_csv('metrics_lstm_tuned.csv', index=False)
#tunes_df.to_csv('tuning_results_long.csv', index=False)
#preds_df.to_csv('predictions_lstm_tuned.csv', index=False)

Country: Austria
Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 374ms/step - loss: 0.0787 - val_loss: 0.5114
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - loss: 0.0710 - val_loss: 0.4831
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - loss: 0.0649 - val_loss: 0.4555
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 0.0644 - val_loss: 0.4284
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - loss: 0.0550 - val_loss: 0.4022
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 0.0463 - val_loss: 0.3768
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - loss: 0.0429 - val_loss: 0.3523
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - loss: 0.0379 - val_loss: 0.3288
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - loss: 0.0335 - val_loss: 0.3062
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - loss: 0.0378 - val_loss: 0.2846
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 0.0287 - val_loss: 0.2640
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - los

In [ ]:
print("Median Training MAPE:", metrics_df["Median_Training_MAPE"].iloc[0] if not metrics_df.empty else np.nan)
metrics_df


Median Training MAPE: 0.11558115663062274


,Country,target_col,MAPE,Training_MAPE,n_train,n_test,best_lookback,best_units,best_dropout,n_val_residuals,PI_width_1st,PI_width_5th,PI_width_final,Median_Training_MAPE
0,Austria,birth_rate,0.032322,0.185798,52,12,5,4,0.2,11,0.635527,0.635527,0.635527,0.115581
1,Sweden,birth_rate,0.055872,0.081098,52,12,2,16,0.3,11,1.739001,1.739001,1.739001,0.115581
2,Germany,birth_rate,0.082854,0.146197,52,12,4,4,0.1,11,0.790651,0.790651,0.790651,0.115581
3,Belgium,birth_rate,0.100753,0.087113,52,12,5,16,0.2,11,0.992598,0.992598,0.992598,0.115581
4,Luxembourg,birth_rate,0.109198,0.071502,52,12,1,4,0.1,11,1.344522,1.344522,1.344522,0.115581
5,France,birth_rate,0.126738,0.088260,52,12,3,8,0.3,11,0.419633,0.419633,0.419633,0.115581
6,Denmark,birth_rate,0.128038,0.099444,52,12,1,4,0.1,11,1.675921,1.675921,1.675921,0.115581
7,Netherlands,birth_rate,0.154811,0.115581,52,12,5,8,0.3,11,1.746147,1.746147,1.746147,0.115581
8,Finland,birth_rate,0.213354,0.121744,52,12,3,8,0.1,11,0.578460,0.578460,0.578460,0.115581
9,Portugal,birth_rate,0.243554,0.280833,52,12,4,8,0.2,11,1.683232,1.683232,1.683232,0.115581
